# Tiny transformer, by hand

This notebook prints every self-attention calculation for `river bank` and `bank river`. It compares attention with and without positional embeddings so you can see exactly where word order enters the model.

In [1]:
from math import isinf

from tiny_transformer import AttentionTrace, teaching_model

## Display helpers

The model returns every intermediate matrix in an `AttentionTrace`. These helpers round the numbers and make masked scores easy to recognize.

In [2]:
def rounded(matrix: list[list[float]]) -> list[list[float | str]]:
    return [
        ["masked" if isinf(number) else round(number, 3) for number in row]
        for row in matrix
    ]


def show(trace: AttentionTrace) -> None:
    print(f"tokens:  {trace.tokens}")
    print(f"X:       {rounded(trace.inputs)}")
    print(f"Q:       {rounded(trace.queries)}")
    print(f"K:       {rounded(trace.keys)}")
    print(f"V:       {rounded(trace.values)}")
    print(f"QK^T:    {rounded(trace.scores)}")
    print(f"softmax: {rounded(trace.weights)}")
    print(f"output:  {rounded(trace.outputs)}")

## Configure the walkthrough

Set `causal` to `True` to prevent each token from attending to tokens that come after it.

In [4]:
causal = True # False
model = teaching_model()

## Compare word order

Without positions, swapping the words only reorders the result. With positions, each word receives a different input vector—and therefore different queries, keys, and values—depending on where it appears.

In [5]:
for use_positions in (False, True):
    label = "WITH positions" if use_positions else "WITHOUT positions"
    print(f"\n=== {label} ===")
    for tokens in (["river", "bank"], ["bank", "river"]):
        print()
        show(
            model.forward(
                tokens,
                use_positions=use_positions,
                causal=causal,
            )
        )


=== WITHOUT positions ===

tokens:  ['river', 'bank']
X:       [[1.0, 0.0], [0.0, 1.0]]
Q:       [[0.0], [1.0]]
K:       [[1.0], [0.0]]
V:       [[1.0, 0.0], [0.0, 1.0]]
QK^T:    [[0.0, 'masked'], [1.0, 0.0]]
softmax: [[1.0, 0.0], [0.731, 0.269]]
output:  [[1.0, 0.0], [0.731, 0.269]]

tokens:  ['bank', 'river']
X:       [[0.0, 1.0], [1.0, 0.0]]
Q:       [[1.0], [0.0]]
K:       [[0.0], [1.0]]
V:       [[0.0, 1.0], [1.0, 0.0]]
QK^T:    [[0.0, 'masked'], [0.0, 0.0]]
softmax: [[1.0, 0.0], [0.5, 0.5]]
output:  [[0.0, 1.0], [0.5, 0.5]]

=== WITH positions ===

tokens:  ['river', 'bank']
X:       [[1.0, 0.0], [0.25, 0.75]]
Q:       [[0.0], [0.75]]
K:       [[1.0], [0.25]]
V:       [[1.0, 0.0], [0.25, 0.75]]
QK^T:    [[0.0, 'masked'], [0.75, 0.188]]
softmax: [[1.0, 0.0], [0.637, 0.363]]
output:  [[1.0, 0.0], [0.728, 0.272]]

tokens:  ['bank', 'river']
X:       [[0.0, 1.0], [1.25, -0.25]]
Q:       [[1.0], [-0.25]]
K:       [[0.0], [1.25]]
V:       [[0.0, 1.0], [1.25, -0.25]]
QK^T:    [[0.0, 'm

## Try it yourself

Change the second position vector in `teaching_model()` or swap the two coordinates of an embedding, then restart the kernel and run all cells to see which calculations change.